# EXP-2 — YOLOv5n (thí nghiệm so sánh E1)

**Người chạy:** Phúc (Backend & Data). **Ý tưởng:** kiểm tra thế hệ YOLOv5 (bản nano) — vốn nhẹ và phổ biến — có đạt độ chính xác tương đương YOLOv8n trên cùng dữ liệu hay không. Huấn luyện bằng repo YOLOv5 chính thức (không phải API ultralytics), cấu hình lưu trong opt.yaml / hyp.yaml của gói E1.

**Giao thức chung E1 (cả 3 thí nghiệm):** dataset mũ bảo hiểm gốc + split hiện tại (368 train / 65 val), imgsz 640, batch 16, seed 0, 50 epochs, chạy trên Google Colab (GPU Tesla T4) ngày 19/11/2026.

**Baseline để đối chiếu — YOLOv8n:** P 0.793 / R 0.710 / mAP50 0.795 / mAP50-95 0.398 (train 95 epochs — ghi rõ chênh lệch số epochs khi so sánh), độ trễ 7.69 ms/ảnh.

*Notebook này đóng gói từ kết quả chạy thật đã lưu trữ trong gói E1 (results.csv + file cấu hình của chính lần chạy). Cell phân tích bên dưới đọc đúng file results.csv đó; output đã nhúng là kết quả tính từ file lưu trữ.*

In [ ]:
# Chạy trong repo YOLOv5 (github.com/ultralytics/yolov5), lệnh huấn luyện đã dùng:
# python train.py --weights yolov5n.pt --data /content/data_helmet.yaml --epochs 50 --batch-size 16 --imgsz 640 --seed 0

In [1]:
import pandas as pd

# File kết quả của chính lần chạy (lưu trong gói E1). results.csv của YOLOv5 đánh số epoch từ 0.
df = pd.read_csv('results.csv')
df.columns = [c.strip() for c in df.columns]
cols = ['metrics/precision', 'metrics/recall', 'metrics/mAP50', 'metrics/mAP50-95']
best = df.loc[df['metrics/mAP50'].idxmax()]
last = df.iloc[-1]
print('Epoch tốt nhất (theo mAP50):', int(best['epoch']) + 1)
print('  P / R / mAP50 / mAP50-95:', [round(float(best[c]), 3) for c in cols])
print('Epoch cuối:', int(last['epoch']) + 1)
print('  P / R / mAP50 / mAP50-95:', [round(float(last[c]), 3) for c in cols])

Epoch tốt nhất (theo mAP50): 50
  P / R / mAP50 / mAP50-95: [0.556, 0.714, 0.679, 0.336]
Epoch cuối: 50
  P / R / mAP50 / mAP50-95: [0.556, 0.714, 0.679, 0.336]


## Kết quả chính thức

| Chỉ số | YOLOv5n (EXP-2) | YOLOv8n (baseline) |
|---|---|---|
| Precision | 0.556 | 0.793 |
| Recall | 0.714 | 0.710 |
| mAP50 | 0.679 | 0.795 |
| mAP50-95 | 0.336 | 0.398 |
| Độ trễ (T4, batch 1, imgsz 640) | 9.22 ms/ảnh (~108 FPS) | 7.69 ms/ảnh (~130 FPS) |

*Độ trễ đo trên 28 ảnh val, cùng điều kiện cho cả 4 model.*

**Kết luận:** mAP50 thấp hơn baseline rõ rệt (−0.116), Precision đặc biệt thấp (0.556) → **loại**. YOLOv5n là thí nghiệm duy nhất bị loại vì độ chính xác, không phải vì tốc độ.